# Part II · Lesson 6 — Remember a student's selected topics

### Import the graph helpers

In [ ]:
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END

### Import the model and messages

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

### Connect to our existing Groq model

In [ ]:
# Keep GROQ_API_KEY in the local .env file.
load_dotenv()
model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
    reasoning_effort="low",
)

### Describe the student's context

In [ ]:
class StudyState(TypedDict):
    student_id: str
    level: str
    topics: list[str]
    messages: Annotated[list, add_messages]
    last_answer: str

### Answer using the restored context

In [ ]:
def tutor(state: StudyState) -> dict:
    reply = model.invoke([
        SystemMessage(
            f"You are a concise Python tutor for a {state['level']} student. "
            f"Selected topics, in order: {', '.join(state['topics'])}. "
            "Use the conversation to understand follow-up references."
        ),
        *state["messages"],
    ])
    return {"messages": [reply], "last_answer": reply.text}

### Attach a checkpointer to the graph

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
builder = StateGraph(StudyState)
builder.add_node("tutor", tutor)
builder.add_edge(START, "tutor")
builder.add_edge("tutor", END)
graph = builder.compile(checkpointer=memory)

### Start the first student's conversation

In [ ]:
student1_config = {"configurable": {"thread_id": "student1"}}
first_response = graph.invoke({
    "student_id": "student1", "level": "beginner",
    "topics": ["lists", "tuples"], "last_answer": "",
    "messages": [HumanMessage("I want to understand Python lists and tuples.")],
}, student1_config)
print(first_response["last_answer"])

### Continue with only the new messages

In [ ]:
for question in [
    "Explain the first one with an example.",
    "How is it different from the second one?",
    "Give me three practice questions on both.",
]:
    response = graph.invoke(
        {"messages": [HumanMessage(question)]}, student1_config
    )
    print("Student:", question)
    print("Tutor:", response["last_answer"])

### Start a different student's thread

In [ ]:
student2_config = {"configurable": {"thread_id": "student2"}}
graph.invoke({
    "student_id": "student2", "level": "beginner",
    "topics": ["loops", "functions"], "last_answer": "",
    "messages": [HumanMessage("I want to understand Python loops and functions.")],
}, student2_config)
second_student = graph.invoke(
    {"messages": [HumanMessage("Explain the first one.")]}, student2_config
)
print(second_student["last_answer"])

### Inspect the stored state

In [ ]:
saved_student1 = graph.get_state(student1_config).values
saved_student2 = graph.get_state(student2_config).values
print(saved_student1["topics"], len(saved_student1["messages"]))
print(saved_student2["topics"], len(saved_student2["messages"]))

## Updating selected topics

### Change topics without discarding the conversation

In [ ]:
changed_topics = graph.invoke({
    "topics": ["dictionaries"],
    "messages": [HumanMessage("Switch to dictionaries. Explain this new topic simply.")],
}, student1_config)
print(changed_topics["topics"])
print(len(changed_topics["messages"]))

## Why add_messages is more than list addition

### Demonstrate an update by message ID

In [ ]:
from langchain_core.messages import AIMessage

earlier = [AIMessage(content="Earlier explanation.", id="reply-1")]
corrected = AIMessage(content="Corrected explanation.", id="reply-1")
merged = add_messages(earlier, [corrected])
print(len(merged), merged[0].text)